In [ ]:
import os
import certifi
import requests

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import Tool


In [7]:
from langchain.agents import create_react_agent,AgentExecutor

In [8]:
# ==========================
# LOAD ENV VARIABLES
# ==========================

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [9]:
search_tool = TavilySearchResults(max_results=3, api_key=TAVILY_API_KEY)

In [10]:
result = search_tool.invoke("Latest News in india")
result

[{'url': 'https://www.thehindu.com/news/national',
  'content': '### Varanasi police book MPs Rahul Gandhi, Pappu Yadav and Awadesh Prasad for \'insulting Sanatan Dharma\'\n\nActivist Tehseen Punaawala was allegedly placed under house arrest ahead of his proposed solo march and hunger strike against the BJP-led Centre\'s E20 fuel policy, at Greater Kailash, New Delhi, Saturday, August 1, 2026.\n\n### Activist Tehseen Poonawalla alleges house arrest ahead of march against E20 fuel policy\n\nAll India Students’ Association president Neha Bora during an interview with PTI, in New Delhi, on July 31, 2026.\n\n### AISA president Neha calls for focus on police accountability amid protest language row\n\nJustice Ujjal Bhuyan. Photo: Special Arrangement\n\n### Supreme Court Collegium’s unexplained recommendations risk bad appointments: Justice Bhuyan [...] August 1, 2026e-Paper\n\nThe Hindu Logo\n\n# India\n\n### Congress says PM Modi owes apology to youth, not ‘forgiveness’\n\n### Security tig

In [12]:
# ==========================
# LLM
# ==========================

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    temperature=0
)

In [13]:
response = llm.invoke("What is AI?")

print(response.content)

**Artificial Intelligence (AI)** is a field of computer science that focuses on creating intelligent machines that can think, learn, and act like humans. AI involves the development of algorithms, statistical models, and computer programs that enable machines to perform tasks that typically require human intelligence, such as:

1. **Reasoning**: Drawing inferences, making decisions, and solving problems.
2. **Learning**: Improving performance on a task over time through experience and data.
3. **Perception**: Interpreting and understanding data from sensors, such as images, speech, and text.
4. **Natural Language Processing (NLP)**: Understanding, generating, and processing human language.

AI systems can be categorized into two main types:

1. **Narrow or Weak AI**: Designed to perform a specific task, such as image recognition, language translation, or playing chess.
2. **General or Strong AI**: A hypothetical AI system that possesses human-like intelligence, reasoning, and problem-s

In [14]:
prompt=hub.pull("hwchase17/react")

c:\Users\katha\OneDrive\Desktop\langchain\.venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [15]:
tools=[search_tool]

In [16]:
# ==========================
# CREATE AGENT
# ==========================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
# ==========================
# EXECUTOR
# ==========================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
response = agent_executor.invoke(
    {
        "input": "Who is the current Prime Minister of India?"
    }
)

print(response["output"])